In [ ]:
# -------------------------------------------------------------------------------------------------
# 📂 1. IMPORTS AND ENVIRONMENT SETUP
# -------------------------------------------------------------------------------------------------
import os
import subprocess
import re
# --- Environment Configuration ---
print(f"✅ Script is running with Process ID (PID): {os.getpid()}")
# -------------------------------------------------------------------------------------------------
#  2. GPU SETUP
# -------------------------------------------------------------------------------------------------
def get_gpu_with_max_free_memory(num_gpus=4):
    """Return the index of the GPU with the most free memory among the first num_gpus devices."""
    try:
        # Query free memory for each GPU using nvidia-smi
        command = ['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits']
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, text=True)
        free_memories = [int(x) for x in re.findall(r'\d+', result.stdout)]
        # Only consider up to num_gpus
        free_memories = free_memories[:num_gpus]
        # Find the GPU index with the most free memory
        max_index = max(range(len(free_memories)), key=lambda i: free_memories[i])
        for idx, free_mb in enumerate(free_memories):
            print(f"                 GPU {idx}:  {free_mb} MB free VRAM")
        return max_index
    except Exception as e:
        print(f"Failed to get GPU info: {e}")
        return 0  # fallback to GPU 0

# Usage example:
best_gpu = get_gpu_with_max_free_memory(4)
os.environ['CUDA_VISIBLE_DEVICES'] = str(best_gpu)
print(f"Set to GPU {best_gpu} with the most free VRAM")
# -------------------------------------------------------------------------------------------------
# --- Standard Library Imports ---
import sys
import json
import time
import difflib # For similarity ratios
import gc

# --- Third-Party Library Imports ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from opencc import OpenCC         # For Simplified to Traditional Chinese conversion
from spellchecker import SpellChecker # For English spelling correction

# -------------------------------------------------------------------------------------------------
# ⚙️ 3. CONFIGURATION AND GLOBAL CONSTANTS
# -------------------------------------------------------------------------------------------------
# --- Model and Path Configuration ---
base_model_path = "./models/Qwen2.5-3B-Instruct" # Ensure this path is correct
adapter_path = "./Qwen2.5-3B-Instruct_Address_Formatter" # Ensure this path is correct
address_test_file_path = './data/test_data_1.txt'

# --- Inference Parameters ---
MAX_NEW_TOKENS = 100
BATCH_SIZE = 128

# --- Post-Processing & Correction Constants ---
CHARS_TO_STRIP_BEFORE_COMPARISON = r" ,'\-;:(/)[\]{}<>?!@#$%^&*`~\|+=-_。，"
PROTECTED_SINGLE_CHARS = ["涌", "湧", "后", "後", "台", "臺"]

# -------------------------------------------------------------------------------------------------
# 🚀 4. INITIALIZATION: MODEL, TOKENIZER, AND UTILITIES
# -------------------------------------------------------------------------------------------------
print("\n--- 🚀 Initializing Model, Tokenizer, and Utilities ---")

# --- Tokenizer ---
print("   - Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# --- Model Quantization ---
print("   - Configuring 4-bit Quantization...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# --- Base Model ---
print("   - Loading Base Model with Quantization...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=quantization_config,
    device_map={"": 0},
    local_files_only=True
)

# --- Fine-tuned PEFT Model ---
print(f"Loading LoRA adapters from {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("✅ Fine-tuned model ready!")


# --- Utility Initializations ---
cc = OpenCC('s2t.json')
print("✅ OpenCC (Simplified to Traditional Chinese) converter ready!")
spell = SpellChecker()
print("✅ English SpellChecker ready!")

# -------------------------------------------------------------------------------------------------
# 🛠️ 5. HELPER FUNCTIONS
# -------------------------------------------------------------------------------------------------

def protect_single_chars(text, char_list):
    # Replace each single char with a unique placeholder
    protected = {}
    for idx, ch in enumerate(char_list):
        if ch in text:
            placeholder = f"__PCHAR_{idx}__"
            text = text.replace(ch, placeholder)
            protected[placeholder] = ch
    return text, protected

def unprotect_single_chars(text, protected):
    for placeholder, ch in protected.items():
        text = text.replace(placeholder, ch)
    return text

def opencc_with_identity_for_protected_chars(text):
    # Protect special chars from OpenCC
    text, protected = protect_single_chars(text, PROTECTED_SINGLE_CHARS)
    text = cc.convert(text)
    text = unprotect_single_chars(text, protected)
    return text

def detect_language(text, chinese_threshold=0.6):
    chinese_chars = 0
    english_chars = 0
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            chinese_chars += 1
        elif 'a' <= char.lower() <= 'z':
            english_chars += 1
    total_relevant_chars = chinese_chars + english_chars
    if total_relevant_chars == 0:
        return 'unknown'
    chinese_ratio = chinese_chars / total_relevant_chars
    if chinese_ratio >= chinese_threshold:
        return 'chinese'
    else:
        return 'english'

def extract_english_alphanumeric_words(text):
    # Extract sequences of English letters (words)
    return re.findall(r'[a-zA-Z]+', text)

def check_and_report_english_spelling_deviations(original_input_english_words, model_output_english_words):
    original_word_set_lower = {word.lower() for word in original_input_english_words}

    deviations = []
    processed_model_words = set()

    for model_word_raw in model_output_english_words:
        if not model_word_raw:
            continue

        model_word_lower = model_word_raw.lower()

        if model_word_lower in processed_model_words:
            continue
        processed_model_words.add(model_word_lower) # Mark as processed

        # If it's an exact match (case-insensitive), it's not a deviation
        if model_word_lower in original_word_set_lower:
            continue

        # If not an exact match, check for genuine misspellings or new words
        correction = spell.correction(model_word_lower)
        if correction and correction != model_word_lower: # If spellchecker found a suggestion
            deviations.append({
                'original_form': model_word_raw,
                'type': 'Genuine Misspelling',
                'suggestion': correction
            })
        else: 
            # No spellchecker suggestion, or it's already "correct" but new
            deviations.append({
                'original_form': model_word_raw,
                'type': 'New/Unmatched Word',
                'note': "Not found in original input."
            })

    return deviations

def robust_insert_with_space(base_string, insert_string, index, is_chinese):
    """
    Inserts a string into a base string at a given index, intelligently adding spaces for non-Chinese text.
    It avoids adding a space if the insertion is happening next to punctuation or at the start/end of the string.
    """
    if is_chinese:
        return base_string[:index] + insert_string + base_string[index:]

    left_part = base_string[:index]
    right_part = base_string[index:]

    # Check if we need a space before the insert_string
    if (left_part and insert_string and
        left_part[-1].isalnum() and insert_string[0].isalnum()):
        left_part += ' '

    # Check if we need a space after the insert_string
    if (right_part and insert_string and
        insert_string[-1].isalnum() and right_part[0].isalnum()):
        right_part = ' ' + right_part

    return left_part + insert_string + right_part

def consume_chars_from_pool_with_index(output_line_raw, char_pool, line_tag, input_lang_type):
    """
    Consumes characters from the pool, marking them as used and storing their output index.
    Returns:
    1. The string formed by characters from the model's output_line_raw that successfully matched the input pool.
    2. A list of characters from model's output_line_raw that *did not* find a match in the input pool.
    """
    matched_chars_in_output = [] # Chars from model's output_line_raw that successfully matched pool
    unmatched_chars_from_model_output = [] # Chars from model's output_line_raw that did NOT match pool

    current_pool_search_start_idx = 0

    for out_char_output_idx, out_char_raw in enumerate(output_line_raw):
        # Determine the character to use for matching against the pool.
        out_char_for_matching = opencc_with_identity_for_protected_chars(out_char_raw) if input_lang_type == 'chinese' else out_char_raw
        out_key_char = out_char_for_matching.lower() if out_char_for_matching.isalpha() else out_char_for_matching

        found_in_pool = False

        # Attempt 1: Search contiguously from last match point for efficiency
        for pool_idx_attempt_1 in range(current_pool_search_start_idx, len(char_pool)):
            pool_item = char_pool[pool_idx_attempt_1]
            if pool_item['used_by'] is None and pool_item['key_char'] == out_key_char:
                pool_item['used_by'] = line_tag
                pool_item['matched_output_idx'] = out_char_output_idx
                matched_chars_in_output.append(out_char_raw) # Append the *raw* char from model output that matched
                found_in_pool = True
                current_pool_search_start_idx = pool_idx_attempt_1 + 1 # Advance search start
                break

        # Attempt 2: If not found contiguously, search the whole pool (non-contiguous match)
        if not found_in_pool:
            for pool_idx_attempt_2 in range(len(char_pool)):
                pool_item = char_pool[pool_idx_attempt_2]
                if pool_item['used_by'] is None and pool_item['key_char'] == out_key_char:
                    pool_item['used_by'] = line_tag
                    pool_item['matched_output_idx'] = out_char_output_idx
                    matched_chars_in_output.append(out_char_raw) # Append the *raw* char from model output that matched
                    found_in_pool = True
                    break

        if not found_in_pool:
            # If no match in the pool, this character from the model's raw output is 'extra'
            unmatched_chars_from_model_output.append(out_char_raw)

    return "".join(matched_chars_in_output).strip(), unmatched_chars_from_model_output


# -------------------------------------------------------------------------------------------------
# 🧠 6. MAIN PROCESSING FUNCTION
# -------------------------------------------------------------------------------------------------

def format_addresses_with_finetuned_model_batch(address_texts_batch):

     # ==================== STAGE A: BATCH PREPARATION AND MODEL INFERENCE ====================
    system_prompt = """
    You are an address formatting assistant. You always return the formatted address with 'Line 1:' and 'Line 2:'.
    Do not include any notes, explanations, or additional conversational turns (e.g., 'user:', 'assistant:').
    Determine the best split point between two components that results in two lines of roughly equal length.
    Strictly output only the formatted lines.
    """

    if not address_texts_batch:
        return [], [], [], []

    prompts_for_batch = []
    actual_prompt_lengths = []
    language_types = []
    original_input_for_char_pooling = []
    original_input_english_words_list = []

    for address_text in address_texts_batch:
        lang_type = detect_language(address_text)
        language_types.append(lang_type)

        if lang_type == 'chinese':
            original_input_processed = opencc_with_identity_for_protected_chars(address_text)
            original_input_english_words_list.append([])
        else:
            original_input_processed = address_text
            original_input_english_words_list.append(extract_english_alphanumeric_words(address_text))

        original_input_for_char_pooling.append(original_input_processed)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": address_text},
        ]
        prompt_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts_for_batch.append(prompt_string)
        tokenized_prompt = tokenizer(prompt_string, return_tensors="pt", add_special_tokens=True).input_ids
        actual_prompt_lengths.append(tokenized_prompt.shape[1])

    model_inputs = tokenizer(prompts_for_batch, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
        )

    all_batch_results = []
     # ==================== STAGE B: PROCESS EACH ITEM IN THE BATCH ====================
    for i, output_sequence in enumerate(outputs):
         # --- B.1: Initialization for this item ---
        original_input_address = address_texts_batch[i]
        original_input_processed_for_char_pool = original_input_for_char_pooling[i]
        input_lang_type = language_types[i]

        original_char_pool = []
        for char_idx, char in enumerate(original_input_processed_for_char_pool):
            original_char_pool.append({
                'char': char,
                'key_char': char.lower() if char.isalpha() else char,
                'used_by': None, # Can be 'line1', 'line2', or None
                'original_idx': char_idx,
                'matched_output_idx': -1 # Index in the output line where it was matched
            })

        generated_tokens = output_sequence[actual_prompt_lengths[i]:]
        raw_response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        cleaned_response_for_parsing = raw_response_text.strip()
        cleaned_response_for_parsing = re.sub(r'^(assistant|user|system)\s*\n?', '', cleaned_response_for_parsing, flags=re.IGNORECASE | re.MULTILINE).strip()
        cleaned_response_for_parsing = re.sub(r'\n+', '\n', cleaned_response_for_parsing).strip()

        match_pattern = r'Line 1:\s*(.*?)\s*\nLine 2:\s*(.*)'
        matches = re.findall(match_pattern, cleaned_response_for_parsing, re.DOTALL)

        line1_content_final = ""
        line2_content_final = ""
        current_system_warning = None
        extra_chars_from_model_output_raw = []

        # --- First Pass: Extra characters ---
        if matches:
            extracted_line1_raw = matches[-1][0].strip()
            extracted_line2_raw = matches[-1][1].strip()

            line1_matched_part, line1_unmatched_chars = consume_chars_from_pool_with_index(extracted_line1_raw, original_char_pool, 'line1', input_lang_type)
            line2_matched_part, line2_unmatched_chars = consume_chars_from_pool_with_index(extracted_line2_raw, original_char_pool, 'line2', input_lang_type)

            extra_chars_from_model_output_raw.extend(line1_unmatched_chars)
            extra_chars_from_model_output_raw.extend(line2_unmatched_chars)

            line1_content_final = line1_matched_part
            line2_content_final = line2_matched_part
        else:
            line1_content_final = ""
            line2_content_final = ""
            current_system_warning = f"Expected 'Line 1: Line 2:' pattern not found in model output. Raw content parsed: '{cleaned_response_for_parsing}'"
            if cleaned_response_for_parsing:
                extra_chars_from_model_output_raw.extend(list(cleaned_response_for_parsing))

        # --- Second Pass: Missing chatacters and Re-insert them ---
        all_categorized_segments = []
        current_group_chars, current_group_start_idx = [], -1

        # Find contiguous blocks of unused characters (omissions)
        for item_idx, item in enumerate(original_char_pool):
            if item['used_by'] is None:
                if not current_group_chars: current_group_start_idx = item['original_idx']
                current_group_chars.append(item['char'])
            else:
                if current_group_chars:
                    group_str = "".join(current_group_chars).strip(CHARS_TO_STRIP_BEFORE_COMPARISON)
                    if group_str:
                        all_categorized_segments.append({'text': group_str, 'start_idx': current_group_start_idx, 'end_idx': item['original_idx'] -1, 'type': 'omission', 'placed': False})
                    current_group_chars, current_group_start_idx = [], -1

        # Handle any trailing group of omissions at the end of the input
        if current_group_chars:
            group_str = "".join(current_group_chars).strip(CHARS_TO_STRIP_BEFORE_COMPARISON)
            if group_str:
                all_categorized_segments.append({'text': group_str, 'start_idx': current_group_start_idx, 'end_idx': len(original_char_pool) - 1, 'type': 'omission', 'placed': False})

        all_categorized_segments.sort(key=lambda x: x['start_idx'])
        current_line1_string = line1_content_final
        current_line2_string = line2_content_final
        is_chinese = (input_lang_type == 'chinese')

        if not is_chinese:
            # For English, re-insert based on context
            omission_segments = [seg for seg in all_categorized_segments if seg['type'] == 'omission']
            insertions_to_perform = []

            for seg_idx, segment_detail in enumerate(omission_segments):
                missing_str = segment_detail['text']
                original_start_idx = segment_detail['start_idx']
                original_end_idx = segment_detail['end_idx']

                if len(missing_str) == 1 and missing_str.isalpha():
                    continue
                
                left_context_item = next((item for item in reversed(original_char_pool[:original_start_idx]) if item['used_by'] in ['line1', 'line2']), None)
                right_context_item = next((item for item in original_char_pool[original_end_idx + 1:] if item['used_by'] in ['line1', 'line2']), None)

                placed_by_context = False

                # Prefer right context for insertion point
                if right_context_item:
                    target_line = right_context_item['used_by']
                    line_str = current_line1_string if target_line == 'line1' else current_line2_string
                    try:
                        right_char_occurrences = [m.start() for m in re.finditer(re.escape(right_context_item['char']), line_str)]
                        if right_char_occurrences:
                            insert_pos = min(right_char_occurrences, key=lambda x: abs(x - right_context_item['matched_output_idx']))
                            insertions_to_perform.append((target_line, insert_pos, missing_str))
                            placed_by_context = True
                    except (ValueError, IndexError):
                        pass

                # Fallback to left context
                if not placed_by_context and left_context_item:
                    target_line = left_context_item['used_by']
                    line_str = current_line1_string if target_line == 'line1' else current_line2_string
                    try:
                        left_char_occurrences = [m.start() for m in re.finditer(re.escape(left_context_item['char']), line_str)]
                        if left_char_occurrences:
                            pos = min(left_char_occurrences, key=lambda x: abs(x - left_context_item['matched_output_idx']))
                            insert_pos = pos + 1
                            insertions_to_perform.append((target_line, insert_pos, missing_str))
                            placed_by_context = True
                    except (ValueError, IndexError):
                        pass

                if placed_by_context:
                    segment_detail['placed'] = True

            # Sort insertions to apply them from end to start, avoiding index shifts
            insertions_to_perform.sort(key=lambda x: (x[0], x[1]), reverse=True)

            for line_name, index, text in insertions_to_perform:
                if line_name == 'line1':
                    current_line1_string = robust_insert_with_space(current_line1_string, text, index, is_chinese)
                elif line_name == 'line2':
                    current_line2_string = robust_insert_with_space(current_line2_string, text, index, is_chinese)

        else: 
            # For Chinese, use a more deterministic insertion logic
            insertions_to_perform = []

            for seg_idx, seg in enumerate(all_categorized_segments):
                if seg['type'] == 'omission' and not seg['placed']:
                    missing_str, start_idx, end_idx = seg['text'], seg['start_idx'], seg['end_idx']
                    placed = False

                    left_context = next((item for item in reversed(original_char_pool[:start_idx]) if item['used_by'] in ['line1', 'line2']), None)
                    right_context = next((item for item in original_char_pool[end_idx+1:] if item['used_by'] in ['line1', 'line2']), None)

                    # Case 1: Sandwiched between two chars that went to the same line
                    if left_context and right_context and left_context['used_by'] == right_context['used_by']:
                        if left_context['matched_output_idx'] < right_context['matched_output_idx']:
                            insertions_to_perform.append((left_context['used_by'], left_context['matched_output_idx'] + 1, missing_str, start_idx))
                            placed = True

                    # Case 2: At the beginning of a line
                    if not placed and right_context and right_context['matched_output_idx'] == 0:
                        insertions_to_perform.append((right_context['used_by'], 0, missing_str, start_idx))
                        placed = True

                    # Case 3: At the end of a line
                    if not placed and left_context:
                        target_line = left_context['used_by']
                        line_len = len(current_line1_string) if target_line == 'line1' else len(current_line2_string)
                        if left_context['matched_output_idx'] == line_len - 1:
                            insertions_to_perform.append((target_line, line_len, missing_str, start_idx))
                            placed = True

                    # Case 4: General fallback (insert before right context or after left context)
                    if not placed:
                        if right_context:
                            insertions_to_perform.append((right_context['used_by'], right_context['matched_output_idx'], missing_str, start_idx))
                            placed = True
                        elif left_context:
                            target_line = left_context['used_by']
                            line_len = len(current_line1_string) if target_line == 'line1' else len(current_line2_string)
                            insertions_to_perform.append((target_line, line_len, missing_str, start_idx))
                            placed = True

                    if placed: seg['placed'] = True

            # Apply Chinese insertions using an offset-based method to handle index shifts
            insertions_to_perform.sort(key=lambda x: (0 if x[0] == 'line1' else 1, x[3]))
            final_line1_chars, final_line2_chars = list(current_line1_string), list(current_line2_string)
            offset1, offset2 = 0, 0

            for line_name, base_idx, text, _ in insertions_to_perform:
                if line_name == 'line1':
                    actual_idx = base_idx + offset1
                    final_line1_chars[actual_idx:actual_idx] = list(text)
                    offset1 += len(text)
                elif line_name == 'line2':
                    actual_idx = base_idx + offset2
                    final_line2_chars[actual_idx:actual_idx] = list(text)
                    offset2 += len(text)
            current_line1_string = "".join(final_line1_chars)
            current_line2_string = "".join(final_line2_chars)

        line1_content_final = current_line1_string.strip()
        line2_content_final = current_line2_string.strip()

        # --- Third Pass: Language-specific checks and finalization ---
        spelling_deviations = []
        if input_lang_type == 'english':
            model_output_english_words = extract_english_alphanumeric_words(line1_content_final + " " + line2_content_final)
            spelling_deviations = check_and_report_english_spelling_deviations(
                original_input_english_words_list[i],
                model_output_english_words
            )

        # --- Collect Additional Output Messages ---
        additional_output_messages = []
        original_output_lines_were_empty = (not line1_content_final and not line2_content_final)
        should_report_omissions = True

        # Handle cases where the model failed completely
        if original_output_lines_were_empty:
            if input_lang_type == 'chinese':
                line1_content_final = opencc_with_identity_for_protected_chars(original_input_address)
                additional_output_messages.append("⚠️ Fallback: No formatted output after all processing, original input placed in Line 1 (Traditional Chinese).")
            else:
                line2_content_final = original_input_address
                additional_output_messages.append("⚠️ Fallback: No formatted output after all processing, original input placed in Line 2 (English/Unknown).")
            should_report_omissions = False

        # Ensure final Chinese output is in Traditional characters
        if input_lang_type == 'chinese':
            line1_content_final = opencc_with_identity_for_protected_chars(line1_content_final)
            line2_content_final = opencc_with_identity_for_protected_chars(line2_content_final)

        additional_output_messages.append(f"Detected Language: {input_lang_type.capitalize()}")

        if current_system_warning and not original_output_lines_were_empty:
             additional_output_messages.append(f"⚠️ System Warning: {current_system_warning}")

        if extra_chars_from_model_output_raw and should_report_omissions:
            unique_extra_chars = sorted(list(set(extra_chars_from_model_output_raw)))
            additional_output_messages.append(f"⚠️ Model generated extra raw characters (not in original input): '{''.join(unique_extra_chars)}'")

        if should_report_omissions:
            omitted_groups = [seg['text'] for seg in all_categorized_segments if seg['type'] == 'omission']
            if omitted_groups:
                additional_output_messages.append(f"💡 Original omissions identified: {', '.join(omitted_groups)}")
                unplaced_omitted_groups = [seg['text'] for seg in all_categorized_segments if seg['type'] == 'omission' and not seg['placed']]
                if not unplaced_omitted_groups:
                    additional_output_messages.append("✅ All identified omissions successfully re-inserted.")
                else:
                    additional_output_messages.append(f"❌ Failed to re-insert some omitted segments: {', '.join(unplaced_omitted_groups)}")

        if input_lang_type == 'english':
            if not spelling_deviations:
                additional_output_messages.append("✅ English word-level deviation check: No significant deviations found.")
            else:
                additional_output_messages.append(f"❌ English word-level deviations found:")
                for dev in spelling_deviations:
                    if dev['type'] == 'Genuine Misspelling':
                        additional_output_messages.append(f"  - Genuine Misspelling: '{dev['original_form']}' -> Suggested: '{dev['suggestion']}'")
                    elif dev['type'] == 'New/Unmatched Word':
                        additional_output_messages.append(f"  - New/Unmatched Word: '{dev['original_form']}' ({dev['note']})")
        else:
            additional_output_messages.append("ℹ️ English word-level deviation check skipped (Input detected as Chinese).")

        all_batch_results.append((line1_content_final, line2_content_final, additional_output_messages, current_system_warning))

    return [r[0] for r in all_batch_results], [r[1] for r in all_batch_results], \
           [r[2] for r in all_batch_results], [r[3] for r in all_batch_results]

# -------------------------------------------------------------------------------------------------
# 📊 7. SCRIPT EXECUTION AND REPORTING
# -------------------------------------------------------------------------------------------------
print("\n--- 📊 Starting Inference and Report Generation ---")

# --- Load Test Data ---
test_data = []
# -------------------------------------------------------------------------------------------------
# for .txt file 
# -------------------------------------------------------------------------------------------------

try:
    with open(address_test_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split('|')
            if len(parts) == 3:
                test_data.append({
                    'original_input': parts[0].strip(),
                    'expected_line1': parts[1].strip(),
                    'expected_line2': parts[2].strip(),
                    'line_num': line_num + 1
                })
            else:
                print(f"Skipping malformed line {line_num + 1} in '{address_test_file_path}': '{line}' (Expected 'input | line1 | line2')")
    print(f"\nSuccessfully loaded {len(test_data)} test cases from '{address_test_file_path}'.")
except FileNotFoundError:
    print(f"\n[ERROR] The file '{address_test_file_path}' was not found.")
    print("Please create this file with format 'input | line1 | line2' (one per line).")
    exit()
# -------------------------------------------------------------------------------------------------
# for .jsonl file
# -------------------------------------------------------------------------------------------------

# try:
#     with open(address_test_file_path, 'r', encoding='utf-8') as f:
#         for line_num, line in enumerate(f, 1):
#             line = line.strip()
#             if not line:
#                 continue
            
#             try:
#                 data = json.loads(line)
#                 # Validate the structure of the JSON object
#                 if 'input' in data and 'output' in data and 'line1' in data['output'] and 'line2' in data['output']:
#                     test_data.append({
#                         'original_input': data['input'],
#                         'expected_line1': data['output']['line1'],
#                         'expected_line2': data['output']['line2'],
#                         'line_num': line_num
#                     })
#                 else:
#                     print(f"Skipping malformed JSON object on line {line_num} in '{address_test_file_path}': Missing required keys.")
#             except json.JSONDecodeError:
#                 print(f"Skipping invalid JSON on line {line_num} in '{address_test_file_path}': '{line}'")
#             except KeyError as e:
#                  print(f"Skipping malformed JSON object on line {line_num}. Missing key: {e}. Content: '{line}'")

#     print(f"\nSuccessfully loaded {len(test_data)} test cases from '{address_test_file_path}'.")
# except FileNotFoundError:
#     print(f"\n[ERROR] The file '{address_test_file_path}' was not found.")
#     print("Please create this file in JSONL format, where each line is a JSON object like:")
#     print('{"input": "...", "output": {"line1": "...", "line2": "..."}}')
#     exit()

print("\n--- INFERENCE AND REPORT GENERATION (BATCHED) ---")
start_overall_time = time.time()

# --- Main Execution Loop ---
total_line1_similarity, total_line2_similarity, total_overall_test_similarity = 0.0, 0.0, 0.0
num_strict_line1_correct, num_strict_line2_correct, num_strict_both_correct = 0, 0, 0
num_no_parsing_cases = 0
total_test_cases = len(test_data)

if total_test_cases > 0:
    for i in range(0, total_test_cases, BATCH_SIZE):
        batch_start_index = i
        batch_end_index = min(i + BATCH_SIZE, total_test_cases)
        current_batch_test_data = test_data[batch_start_index : batch_end_index]
        current_batch_addresses = [item['original_input'] for item in current_batch_test_data]

        batch_inference_start_time = time.time()
        formatted_line1_batch, formatted_line2_batch, additional_messages_batch, system_warnings_batch = \
            format_addresses_with_finetuned_model_batch(current_batch_addresses)
        batch_duration = time.time() - batch_inference_start_time

        print(f"\n--- Processing Batch {i // BATCH_SIZE + 1}/{ -(-total_test_cases // BATCH_SIZE)} (Cases {batch_start_index+1}-{batch_end_index}) ---")
        for j, test_case in enumerate(current_batch_test_data):
            original_addr, expected_l1_orig, expected_l2_orig, line_num = \
                test_case['original_input'], test_case['expected_line1'], test_case['expected_line2'], test_case['line_num']

            model_l1_out, model_l2_out, add_msgs, sys_warn = \
                formatted_line1_batch[j], formatted_line2_batch[j], additional_messages_batch[j], system_warnings_batch[j]

            conversion_note = ""
            input_lang_context = detect_language(original_addr)
            if input_lang_context == 'chinese':
                expected_l1_comp = opencc_with_identity_for_protected_chars(expected_l1_orig)
                expected_l2_comp = opencc_with_identity_for_protected_chars(expected_l2_orig)
                if expected_l1_orig != expected_l1_comp or expected_l2_orig != expected_l2_comp:
                    conversion_note = " (Note: Original expected was Simplified, compared against Traditional)"
            else:
                expected_l1_comp, expected_l2_comp = expected_l1_orig, expected_l2_orig

            print(f"\n--- Test Case {batch_start_index + j + 1} (Line {line_num}) ---")
            print(f"Input: {original_addr}")
            print(f"Expected Line 1: {expected_l1_comp}")
            print(f"Expected Line 2: {expected_l2_comp}")
            if conversion_note: print(f"  {conversion_note}")
            print(f"Model Output Line 1: {model_l1_out}")
            print(f"Model Output Line 2: {model_l2_out}")

            l1_sim = difflib.SequenceMatcher(None, model_l1_out, expected_l1_comp).ratio()
            l2_sim = difflib.SequenceMatcher(None, model_l2_out, expected_l2_comp).ratio()
            overall_sim = (l1_sim + l2_sim) / 2.0
            total_line1_similarity += l1_sim
            total_line2_similarity += l2_sim
            total_overall_test_similarity += overall_sim

            print(f"Line 1 Similarity: {l1_sim*100:.2f}%, Line 2 Similarity: {l2_sim*100:.2f}%")
            print(f"Overall Test Similarity: {overall_sim*100:.2f}%")

            is_l1_correct = (model_l1_out == expected_l1_comp)
            is_l2_correct = (model_l2_out == expected_l2_comp)
            if is_l1_correct: num_strict_line1_correct += 1
            if is_l2_correct: num_strict_line2_correct += 1

            if is_l1_correct and is_l2_correct:
                print("➡️ Result: ✅ EXACT MATCH (Line 1 & Line 2)")
                num_strict_both_correct += 1
            else:
                print(f"➡️ Result: ❌ PARTIAL/NO MATCH (L1 Correct: {is_l1_correct}, L2 Correct: {is_l2_correct})")

            if any("Fallback:" in msg for msg in add_msgs):
                num_no_parsing_cases += 1

            print("--- Detailed Analysis ---")
            for msg in add_msgs: print(f"  {msg}")
        print(f"Batch inference time: {batch_duration:.2f} seconds")
else:
    print("\n--- No addresses found in the file to process. ---")

# --- Final Summary Report ---
end_overall_time = time.time()
total_elapsed_time = end_overall_time - start_overall_time
print(f"\n\n# ================================================================================================")
print(f"#                                   FINAL SUMMARY REPORT")
print(f"# ================================================================================================")
print(f"Total Test Cases Processed: {total_test_cases}")
if total_test_cases > 0:
    avg_time_per_address = total_elapsed_time / total_test_cases
    avg_l1_sim_pct = (total_line1_similarity / total_test_cases) * 100
    avg_l2_sim_pct = (total_line2_similarity / total_test_cases) * 100
    avg_overall_sim_pct = (total_overall_test_similarity / total_test_cases) * 100
    no_parsing_pct = (num_no_parsing_cases / total_test_cases) * 100
    print(f"\nAverage Line 1 Similarity: {avg_l1_sim_pct:.2f}%")
    print(f"Average Line 2 Similarity: {avg_l2_sim_pct:.2f}%")
    print(f"Average Overall Test Similarity: {avg_overall_sim_pct:.2f}%")
    print(f"\nStrict Match Counts:")
    print(f"  Line 1 Strictly Correct: {num_strict_line1_correct}/{total_test_cases}")
    print(f"  Line 2 Strictly Correct: {num_strict_line2_correct}/{total_test_cases}")
    print(f"  Both Lines Strictly Correct: {num_strict_both_correct}/{total_test_cases}")
    print(f"\nNo Address Parsing / Fallback Cases: {num_no_parsing_cases}/{total_test_cases} ({no_parsing_pct:.2f}%)")
print(f"# ================================================================================================")

print(f"\nTotal elapsed time for {total_test_cases} addresses: {total_elapsed_time:.2f} seconds")
if total_test_cases > 0:
    print(f"Average time per address: {avg_time_per_address:.4f} seconds")
print(f"--- Now terminating process {os.getpid()} to release all resources. ---")

# -------------------------------------------------------------------------------------------------
# 🧹 8. VRAM CLEANUP
# -------------------------------------------------------------------------------------------------
print("\n--- 🧹 Releasing GPU VRAM ---")
time.sleep(6)
os.kill(os.getpid(), 9)
print(f"\n--- Now terminating process {os.getpid()} to release all resources. ---")
time.sleep(5) # Give a moment for final prints to flush
print(f"✅ Processing complete!")

✅ Script is running with Process ID (PID): 1779373
                 GPU 0:  4017 MB free VRAM
                 GPU 1:  38 MB free VRAM
                 GPU 2:  1036 MB free VRAM
                 GPU 3:  830 MB free VRAM
Set to GPU 0 with the most free VRAM

--- 🚀 Initializing Model, Tokenizer, and Utilities ---
   - Loading Tokenizer...
   - Configuring 4-bit Quantization...
   - Loading Base Model with Quantization...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapters from ./Qwen2.5-3B-Instruct_Address_Formatter
✅ Fine-tuned model ready!
✅ OpenCC (Simplified to Traditional Chinese) converter ready!
✅ English SpellChecker ready!

--- 📊 Starting Inference and Report Generation ---

[ERROR] The file './data/test_data_1.txt' was not found.
Please create this file with format 'input | line1 | line2' (one per line).

--- INFERENCE AND REPORT GENERATION (BATCHED) ---

--- No addresses found in the file to process. ---


# ================================================================================================
#                                   FINAL SUMMARY REPORT
# ================================================================================================
Total Test Cases Processed: 0
# ================================================================================================

Total elapsed time for 0 addresses: 0.00 seconds
--- Now terminating process 1779373 to release all resources. ---

--- 🧹 Releasing GPU VRA